# Install dependencies

In [1]:
!pip install -q transformers datasets accelerate evaluate rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


# Imports

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import evaluate
import numpy as np
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import random
import json
import re

# Load data

In [3]:
dataset = load_dataset("cnn_dailymail", "3.0.0")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [4]:
N_TRAIN = 10_000
N_VAL   = 2_000
N_TEST  = 2_000

train_dataset = dataset["train"].shuffle(seed=42).select(range(N_TRAIN))
val_dataset   = dataset["validation"].shuffle(seed=42).select(range(N_VAL))
test_dataset  = dataset["test"].shuffle(seed=42).select(range(N_TEST))

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 10000
})
Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 2000
})
Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 2000
})


In [5]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Using device:", device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Using device: cuda


In [6]:
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = ["summarize: " + art for art in examples["article"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["highlights"],
            max_length=max_target_length,
            truncation=True,
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [7]:
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names,
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

print(tokenized_train[0])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4118: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'input_ids': [21603, 10, 938, 3, 5, 11016, 12528, 3, 5, 3, 10744, 8775, 20619, 2326, 10, 3, 5, 10668, 10, 4928, 3, 6038, 6, 204, 1332, 2038, 3, 5, 1820, 3, 5, 3, 6880, 4296, 11430, 10, 3, 5, 12046, 10, 4560, 3, 6038, 6, 204, 1332, 2038, 3, 5, 5245, 724, 13, 8, 337, 384, 113, 3977, 16, 3, 9, 14491, 22133, 45, 4146, 1911, 6778, 15, 14566, 53, 133, 43, 118, 25429, 3, 31, 4065, 77, 676, 31, 6, 16273, 7, 243, 469, 5, 37, 5678, 13, 4464, 1158, 1079, 11, 31423, 6176, 130, 3883, 5815, 70, 3062, 6, 7758, 60, 35, 6, 44, 8, 1156, 234, 79, 2471, 30, 4691, 1635, 109, 1210, 1061, 16, 5184, 12940, 6, 4653, 26334, 5, 37, 16, 10952, 7, 43, 230, 2946, 139, 8, 14319, 336, 1856, 6, 28, 16273, 7, 2145, 8, 386, 3977, 590, 28, 8, 384, 31, 7, 3947, 1782, 6, 13, 4146, 1911, 6778, 15, 14566, 53, 45, 3, 9, 21859, 5, 21902, 447, 10, 37, 16, 10952, 7, 43, 2946, 139, 8, 14319, 13, 386, 724, 13, 8, 337, 384, 113, 130, 435, 16, 70, 14491, 22133, 336, 1851, 5, 1079, 11, 31423, 6176, 33, 3, 22665, 3, 5, 71, 210, 1329,

In [8]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [9]:
training_args = Seq2SeqTrainingArguments(
    output_dir="flan_t5_small_cnn_summarizer",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=7,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=max_target_length,
    report_to="none",
)

In [10]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

/tmp/ipython-input-803467194.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.273600,1.892182
2,2.260300,1.885538
3,2.200500,1.885366


Epoch,Training Loss,Validation Loss
1,2.273600,1.892182
2,2.260300,1.885538
3,2.200500,1.885366
4,2.209700,1.880005
5,2.186100,1.881428
6,2.153100,1.876272
7,2.141300,1.876747


TrainOutput(global_step=4375, training_loss=2.205734340122768, metrics={'train_runtime': 3788.5842, 'train_samples_per_second': 18.477, 'train_steps_per_second': 1.155, 'total_flos': 1.3011966452809728e+16, 'train_loss': 2.205734340122768, 'epoch': 7.0})

In [12]:
save_dir = "flan_t5_small_cnn_summarizer_final"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved fine-tuned model to:", save_dir)

Saved fine-tuned model to: flan_t5_small_cnn_summarizer_final


In [13]:
model.eval()

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [14]:
rouge = evaluate.load("rouge")

test_loader = DataLoader(
    tokenized_test,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator,
)

In [15]:
all_preds = []
all_labels = []

for batch in tqdm(test_loader, desc="Evaluating on test set"):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    labels = batch["labels"].numpy()

    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True,
        )

    decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    labels = np.where(labels == -100, tokenizer.pad_token_id, labels)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    all_preds.extend(decoded_preds)
    all_labels.extend(decoded_labels)

rouge_result = rouge.compute(
    predictions=all_preds,
    references=all_labels,
    use_stemmer=True,
)

print("ROUGE scores:")
for k, v in rouge_result.items():
    print(f"{k}: {v:.4f}")

Evaluating on test set:   0%|          | 0/250 [00:00<?, ?it/s]

ROUGE scores:
rouge1: 0.3842
rouge2: 0.1696
rougeL: 0.2708
rougeLsum: 0.2707


In [16]:
N_EXAMPLES = 5
indices = random.sample(range(len(test_dataset)), N_EXAMPLES)

for i in indices:
    ex = test_dataset[i]
    article = ex["article"]
    reference = all_labels[i]
    prediction = all_preds[i]

    print(f"\n======== EXAMPLE {i} ========")
    print("ARTICLE (truncated):")
    print(article[:800], "...\n")

    print("REFERENCE SUMMARY:")
    print(reference, "\n")

    print("MODEL SUMMARY:")
    print(prediction)
    print("==============================")


======== EXAMPLE 1309 ========
ARTICLE (truncated):
Manchester City and Chelsea are set to battle it out for the signature of West Ham left-back Aaron Cresswell this summer. Cresswell has impressed for the Hammers this campaign and City are desperate to add to their quota of English players with the likes of James Milner on the brink of leaving the club. City will have another chance to run the rule over the 25-year-old when West Ham travel to the Etihad this Sunday. Chelsea and Manchester City are set to battle it out for West Ham left back Aaron Cresswell this summer . The 25-year-old has impressed during his first season in the Premier League since leaving Ipswich . West Ham snapped up Cresswell for £2million from Ipswich last summer but he has adapted to the top tier with relative ease and attracted the eye of the division’s Champions L ...

REFERENCE SUMMARY:
Aaron Cresswell has impressed during debut season in Premier League . The left back joined West Ham from Championship club

In [17]:
!zip -r flan_t5_small_cnn_summarizer_final.zip flan_t5_small_cnn_summarizer_final

  adding: flan_t5_small_cnn_summarizer_final/ (stored 0%)
  adding: flan_t5_small_cnn_summarizer_final/training_args.bin (deflated 53%)
  adding: flan_t5_small_cnn_summarizer_final/model.safetensors (deflated 7%)
  adding: flan_t5_small_cnn_summarizer_final/spiece.model (deflated 48%)
  adding: flan_t5_small_cnn_summarizer_final/generation_config.json (deflated 27%)
  adding: flan_t5_small_cnn_summarizer_final/special_tokens_map.json (deflated 85%)
  adding: flan_t5_small_cnn_summarizer_final/tokenizer.json (deflated 74%)
  adding: flan_t5_small_cnn_summarizer_final/tokenizer_config.json (deflated 95%)
  adding: flan_t5_small_cnn_summarizer_final/config.json (deflated 62%)
